# B. CPU-only: Kaggle Secret and supervised-subprocess isolation

**Accelerator: None** · fresh Kaggle session · Python 3.13 · generated by `scripts/build_kaggle_diagnostic_notebooks.py`

No Stokastisk, no vLLM, no GPU. Fetches the exact reviewed Enigma connector revision and tests, one at a time: secret retrieval (no Enigma request), a child that exits by itself, a child that needs graceful shutdown, sentinel survival, parent PID/PGID/SID preservation, checkpoint writes and ZIP export. No runtime is registered.

**Evidence durability.** Every step prints flushed `MARK` and `VERDICT` lines and appends them to an fsynced `markers.jsonl` in a new, never-reused attempt directory. After a complete platform termination the VM's files may be gone (on 2026-11-26 a new session had to re-clone everything), but cell output printed before the failure was kept in the saved notebook. **Missing files do not prove that no crash occurred.** After a fatal session, copy the last `MARK` lines (UTC time, `t` = seconds since the prelude) into the incident report.

Run the cells top to bottom (or *Run all*). Every step ends with `VERDICT <step>: PASS | FAIL | INCONCLUSIVE`; anything but PASS raises and stops the run. Then run the final **evidence export** cell manually.

In [ ]:
# P0. Prelude: environment guard, attempt id, markers and verdicts. Standard library only: no network, no GPU library is imported.
import os, sys, json, time, uuid, platform, shutil, subprocess, re
NOTEBOOK = "enigma_kaggle_cpu_connector_isolation"
EXPECT = "cpu"                                   # "cpu" = Accelerator None; "gpu" = GPU T4 x2
assert os.path.isdir("/kaggle/working"), "Kaggle only: refusing to run anywhere else (never on a developer laptop)"


class DiagnosticStop(Exception):
    """Raised by every non-PASS verdict so that 'Run all' stops here. Later cells must never run after a failure."""


HAS_GPU_DEVICE = os.path.exists("/dev/nvidia0")
if EXPECT == "cpu" and HAS_GPU_DEVICE:
    raise DiagnosticStop("INCONCLUSIVE: wrong environment. A GPU is attached; set Accelerator = None and start a NEW session")
if EXPECT == "gpu" and not HAS_GPU_DEVICE:
    raise DiagnosticStop("INCONCLUSIVE: wrong environment. No GPU; set Accelerator = GPU T4 x2 and start a NEW session")
assert sys.version_info[:2] == (3, 13), f"expected Python 3.13 (build-5 wheel is cp313), got {platform.python_version()}"

ATTEMPT = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime()) + "-" + uuid.uuid4().hex[:8]
ATTEMPT_DIR = f"/kaggle/working/enigma-diag-attempts/{NOTEBOOK}/{ATTEMPT}"
os.makedirs(ATTEMPT_DIR, exist_ok=False)               # a new directory per attempt: earlier evidence is never overwritten
T0 = time.monotonic()
RESULTS = {}                                           # verdicts of THIS kernel session only (stale outputs can never satisfy a check)


def _read(path):
    try:
        with open(path) as fh:
            return fh.read(65536)
    except OSError:
        return None


def host():
    """Host memory, cgroup v2 memory (current, max, events), shared memory, disk, uptime and load. No environment variables are read."""
    h = {}
    for line in (_read("/proc/meminfo") or "").splitlines():
        k, _, v = line.partition(":")
        if k in ("MemTotal", "MemAvailable", "Shmem"):
            h[k.lower() + "_mib"] = round(int(v.split()[0]) / 1024, 1)
    mib = lambda s: None if s is None or not s.strip().isdigit() else round(int(s) / 2**20, 1)   # noqa: E731 ("max" = unlimited -> None)
    h["cgroup_current_mib"], h["cgroup_max_mib"] = mib(_read("/sys/fs/cgroup/memory.current")), mib(_read("/sys/fs/cgroup/memory.max"))
    h["cgroup_events"] = {k: int(v) for k, _, v in (l.partition(" ") for l in (_read("/sys/fs/cgroup/memory.events") or "").splitlines()) if v.strip().isdigit()}
    for d in ("/kaggle/working", "/tmp", "/dev/shm"):
        try:
            u = shutil.disk_usage(d)
            h["disk_" + d.strip("/").replace("/", "_")] = {"used_gib": round(u.used / 2**30, 2), "free_gib": round(u.free / 2**30, 2)}
        except OSError:
            pass
    up = _read("/proc/uptime")
    h["vm_uptime_s"] = float(up.split()[0]) if up else None
    h["loadavg"] = os.getloadavg()
    return h


def ident():
    return {"pid": os.getpid(), "ppid": os.getppid(), "pgid": os.getpgid(0), "sid": os.getsid(0)}


def mark(name, **fields):
    rec = {"attempt": ATTEMPT, "nb": NOTEBOOK, "marker": name, "utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
           "t": round(time.monotonic() - T0, 2), **ident(), **host(), **fields}
    line = json.dumps(rec, default=str)
    with open(f"{ATTEMPT_DIR}/markers.jsonl", "a") as fh:
        fh.write(line + "\n"); fh.flush(); os.fsync(fh.fileno())
    print("MARK", line, flush=True)
    return rec


def verdict(step, status, reason, **evidence):
    assert status in ("PASS", "FAIL", "INCONCLUSIVE"), status
    RESULTS[step] = {"status": status, "reason": reason, "attempt": ATTEMPT, **evidence}
    mark("verdict", step=step, status=status, reason=reason, **evidence)
    print(f"VERDICT {step}: {status} | {reason}", flush=True)
    if status != "PASS":
        raise DiagnosticStop(f"{step} {status}: {reason}")


def require(*steps):
    """Every prerequisite must have PASSED in this very session (same attempt id)."""
    bad = [s for s in steps if RESULTS.get(s, {}).get("status") != "PASS" or RESULTS[s].get("attempt") != ATTEMPT]
    if bad:
        raise DiagnosticStop(f"INCONCLUSIVE: prerequisite(s) did not PASS in this session: {bad}. Start a new session and run from the top.")


def observe(step, seconds, every=15.0, sample=None):
    """Survival observation: one MARK every `every` s. PASS only if the full duration elapsed with no cgroup OOM kill and no failed sample check."""
    start, oom0, n, problems = time.monotonic(), host()["cgroup_events"].get("oom_kill"), 0, []
    mark(step + "_begin", seconds=seconds, every=every)
    while time.monotonic() - start < seconds:
        time.sleep(min(every, max(0.0, seconds - (time.monotonic() - start))))
        n += 1
        extra = sample() if sample else {}
        if extra.get("problem"):
            problems.append(extra["problem"])
        mark(step + "_alive", i=n, **extra)
    el, oom1 = time.monotonic() - start, host()["cgroup_events"].get("oom_kill")
    delta = None if oom0 is None or oom1 is None else oom1 - oom0
    ok = el >= seconds - 1.0 and delta in (None, 0) and not problems
    verdict(step, "PASS" if ok else "FAIL", f"observed {el:.0f}/{seconds}s, oom_kill delta {delta}, problems {problems[:3]}", observed_s=round(el, 1),
            oom_kill_delta=delta, samples=n)


def export_evidence():
    """Zip this attempt's directory (only it). Safe to run after a failure."""
    base = f"/kaggle/working/enigma-diag-attempts/{NOTEBOOK}-{ATTEMPT}"
    path = shutil.make_archive(base, "zip", ATTEMPT_DIR)
    print("evidence archive:", path, os.path.getsize(path), "bytes", flush=True)
    return path


_os = dict(l.split("=", 1) for l in (_read("/etc/os-release") or "").splitlines() if "=" in l)
_kaggle = {k: os.environ[k] for k in ("KAGGLE_KERNEL_RUN_TYPE", "KAGGLE_DOCKER_IMAGE") if k in os.environ}   # allow-list only; never the raw environment
mark("prelude", python=platform.python_version(), expect=EXPECT, gpu_device=HAS_GPU_DEVICE, os=_os.get("PRETTY_NAME", "").strip('"'), kaggle=_kaggle)
print("ATTEMPT", ATTEMPT, "| markers:", ATTEMPT_DIR + "/markers.jsonl", flush=True)

## B0. Idle survival control: 5 minutes

In [ ]:
observe("B0_idle_5min", 300, every=15)

## B1. Fetch the reviewed connector

In [ ]:
require("B0_idle_5min")
# Fetch the exact reviewed Enigma connector revision (GitHub only; no Enigma request). Verifies the checked-out SHA.
REV = "c5e5e8dedcaa2582f2a55a5918ab3ad6f03b3a49"
repo = f"{ATTEMPT_DIR}/enigma-src"
mark("connector_fetch_begin", rev=REV)
for cmd in (["git", "init", "-q", repo], ["git", "-C", repo, "fetch", "-q", "--depth", "1", "https://github.com/stokastisk-ai/enigma.git", REV],
            ["git", "-C", repo, "checkout", "-q", "FETCH_HEAD"]):
    r = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
    if r.returncode != 0:
        verdict("connector_fetch", "FAIL", f"{cmd[3] if len(cmd) > 3 else cmd[1]} failed rc={r.returncode}", stderr_tail=r.stderr[-500:])
head = subprocess.run(["git", "-C", repo, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
sys.path.insert(0, f"{repo}/connector")
import enigma_connector
from enigma_connector.supervise import ConnectorProcess            # stdlib-only module; run_stage is imported only where a stage runs
ok = head == REV and enigma_connector.__file__.startswith(repo)
verdict("connector_fetch", "PASS" if ok else "FAIL", f"HEAD={head}", head=head, module=enigma_connector.__file__)

## B2. Kaggle Secret retrieval only
`UserSecretsClient` contacts Kaggle's own secret service; nothing is sent to Enigma. Only presence and format are recorded.

In [ ]:
require("connector_fetch")
mark("B2_before_secret_retrieval")
try:
    from kaggle_secrets import UserSecretsClient
    _v = UserSecretsClient().get_secret("ENIGMA_RUNTIME_TOKEN")
except Exception as e:
    mark("B2_after_secret_retrieval", ok=False, error=type(e).__name__)
    verdict("B2_secret", "INCONCLUSIVE", f"secret not readable ({type(e).__name__}): attach ENIGMA_RUNTIME_TOKEN under Add-ons > Secrets")
_fmt = bool(re.fullmatch(r"enig_rt_[a-z0-9]{8}_[A-Za-z0-9_-]{43}", (_v or "").strip()))
del _v
mark("B2_after_secret_retrieval", ok=True, format_ok=_fmt)
verdict("B2_secret", "PASS" if _fmt else "FAIL", "present and well-formed" if _fmt else "present but malformed")

## B3. Supervised child that exits by itself, B4. child that needs graceful SIGTERM
A sentinel in its own session must survive; this kernel's PID/PGID/SID must not change; the self-exiting child must receive **no** signal.

In [ ]:
require("B2_secret")
me = ident()
sentinel = subprocess.Popen([sys.executable, "-c", "import time\nwhile True: time.sleep(0.5)"], start_new_session=True)
child_env = {"PATH": os.environ.get("PATH", "/usr/bin:/bin")}             # minimal environment: no Kaggle or Enigma variables are passed
p1 = ConnectorProcess([], child_env, f"{ATTEMPT_DIR}/b3", command=[sys.executable, "-c", "print('exit by itself')"])
mark("B3_before_spawn"); pid1 = p1.start(); mark("B3_after_spawn", child=pid1)
rc1 = p1.wait(30); mark("B3_child_exit", rc=rc1)
mark("B3_before_cleanup"); r1 = p1.stop(grace_s=2); mark("B3_after_cleanup", rc=r1["returncode"], sent=r1["signals_sent"], refused=r1["signals_refused"])
ok1 = r1["returncode"] == 0 and r1["signals_sent"] == [] and sentinel.poll() is None and ident() == me
verdict("B3_self_exit", "PASS" if ok1 else "FAIL", f"rc={r1['returncode']} signals={r1['signals_sent']} sentinel_alive={sentinel.poll() is None}")

script = "import signal,sys,time\nsignal.signal(signal.SIGTERM, lambda *a: sys.exit(0))\nwhile True: time.sleep(0.1)"
p2 = ConnectorProcess([], child_env, f"{ATTEMPT_DIR}/b4", command=[sys.executable, "-c", script])
mark("B4_before_spawn"); pid2 = p2.start(); mark("B4_after_spawn", child=pid2)
time.sleep(1)
mark("B4_before_cleanup"); r2 = p2.stop(grace_s=10, kill_after_s=5); mark("B4_after_cleanup", rc=r2["returncode"], sent=r2["signals_sent"], escalated=r2["escalated_to_sigkill"])
ok2 = r2["returncode"] == 0 and r2["signals_sent"] == ["SIGTERM"] and not r2["escalated_to_sigkill"] and sentinel.poll() is None and ident() == me
sentinel.kill(); sentinel.wait(5)
verdict("B4_graceful", "PASS" if ok2 else "FAIL", f"rc={r2['returncode']} signals={r2['signals_sent']} escalated={r2['escalated_to_sigkill']}")

## B5. Checkpoint write and B6. ZIP archive

In [ ]:
require("B4_graceful")
mark("B5_before_checkpoint")
data = {"attempt": ATTEMPT, "results": {k: v["status"] for k, v in RESULTS.items()}, "utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
tmp, final = f"{ATTEMPT_DIR}/checkpoint.tmp", f"{ATTEMPT_DIR}/checkpoint.json"
with open(tmp, "w") as fh:
    json.dump(data, fh); fh.flush(); os.fsync(fh.fileno())
os.replace(tmp, final)
mark("B5_after_checkpoint")
verdict("B5_checkpoint", "PASS" if json.load(open(final)) == data else "FAIL", "atomic write and read-back")
import zipfile
mark("B6_before_archive")
path = export_evidence()
with zipfile.ZipFile(path) as z:
    good = z.testzip() is None and "markers.jsonl" in z.namelist() and "checkpoint.json" in z.namelist()
mark("B6_after_archive", archive=path, bytes=os.path.getsize(path))
verdict("B6_archive", "PASS" if good else "FAIL", f"{path}")

## Evidence export

In [ ]:
# Evidence export (run it even after a FAIL / INCONCLUSIVE stop). Review the archive before sharing it; it contains no credential.
mark("export_begin", results={k: v["status"] for k, v in RESULTS.items()})
archive = export_evidence()
print(json.dumps({k: {"status": v["status"], "reason": v["reason"]} for k, v in RESULTS.items()}, indent=1), flush=True)